# 环境配置

In [1]:
!pip install datasets swanlab -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.9/153.9 kB 16.0 MB/s eta 0:00:00


In [2]:
!wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=1a0sf5C209CLW5824TJkUM4olMy0zZWpg' -O fake_sft.json

--2025-12-24 12:10:59--  https://docs.google.com/uc?export=download&id=1a0sf5C209CLW5824TJkUM4olMy0zZWpg
Resolving docs.google.com (docs.google.com)... 74.125.199.138, 74.125.199.102, 74.125.199.101, ...
Connecting to docs.google.com (docs.google.com)|74.125.199.138|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1a0sf5C209CLW5824TJkUM4olMy0zZWpg&export=download [following]
--2025-12-24 12:10:59--  https://drive.usercontent.google.com/download?id=1a0sf5C209CLW5824TJkUM4olMy0zZWpg&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 74.125.135.132, 2607:f8b0:400e:c01::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|74.125.135.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 210335 (205K) [application/octet-stream]
Saving to: ‘fake_sft.json’

fake_sft.json       100%[===================>] 205.41K  --.-KB/s    in 

In [3]:
from datasets import Dataset
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer, GenerationConfig
from peft import LoraConfig, TaskType, get_peft_model
import torch

In [4]:
!ls

fake_sft.json  sample_data


In [ ]:
# 将JSON文件转换为CSV文件
df = pd.read_json('fake_sft.json')
ds = Dataset.from_pandas(df)
ds[:3]

{'system': ['将文本中的name、address、email、question提取出来，以json格式输出，字段为name、address、email、question，值为文本中提取出来的内容。',
  '将文本中的name、address、email、question提取出来，以json格式输出，字段为name、address、email、question，值为文本中提取出来的内容。',
  '将文本中的name、address、email、question提取出来，以json格式输出，字段为name、address、email、question，值为文本中提取出来的内容。'],
 'instruction': ['龙琳：宁夏回族自治区璐市城东林街g座 955491，邮箱 nafan@example.com。小区垃圾堆积成山，晚上噪音扰人清梦，停车难上加难，简直无法忍受！',
  '周飞：海陵通辽路Y座 375698，邮箱 ping23@example.com 。小区垃圾成堆，晚上吵得要命，车位也太少，简直没法住人！',
  '林桂兰：丰都李路V座 458454，邮箱leiwang@example.com。你们小区垃圾成山，噪音扰民，停车位简直是个笑话，这样的管理真是让人火大！'],
 'input': ['', '', ''],
 'output': ['```json\n{\n    "name": "龙琳",\n    "address": "宁夏回族自治区璐市城东林街g座 955491",\n    "email": "nafan@example.com",\n    "question": "小区垃圾堆积成山，晚上噪音扰人清梦，停车难上加难，简直无法忍受！"\n}\n```',
  '```json\n{\n    "name": "周飞",\n    "address": "海陵通辽路Y座",\n    "email": "ping23@example.com",\n    "question": "小区垃圾成堆，晚上吵得要命，车位也太少，简直没法住人！"\n}\n```',
  '```json\n{\n    "name": "林桂兰",\n    "address": "丰都李路V座 458454",\n    "email": "

In [8]:
model_id = 'Qwen/Qwen3-0.6B'

In [ ]:
# 从预训练模型加载对应的分词器
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Qwen2TokenizerFast(name_or_path='Qwen/Qwen3-0.6B', vocab_size=151643, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized

对大语言模型进行 `supervised-finetuning`（`sft`，有监督微调）的数据格式如下：

```json
{
  "instruction": "回答以下用户问题，仅输出答案。",
  "input": "1+1等于几?",
  "output": "2"
}
```

其中，`instruction` 是用户指令，告知模型其需要完成的任务；`input` 是用户输入，是完成用户指令所必须的输入内容；`output` 是模型应该给出的输出。

有监督微调的目标是让模型具备理解并遵循用户指令的能力。因此，在构建数据集时，我们应针对我们的目标任务，针对性构建数据。比如，如果我们的目标是通过大量人物的对话数据微调得到一个能够 role-play 甄嬛对话风格的模型，因此在该场景下的数据示例如下：

```json
{
  "instruction": "你父亲是谁？",
  "input": "",
  "output": "家父是大理寺少卿甄远道。"
}
```

`Qwen3` 采用的 `Chat Template`格式如下：

由于 `Qwen3` 是混合推理模型，因此可以手动选择开启思考模式

不开启 `thinking mode`

In [11]:
from pydoc import text


messages = [
    {"role": "system", "content": "你是一个专业的助手"},
    {"role": "user", "content": "你现在怎么样"},
    {"role": "assistant", "content": "I'm fine, think you. and you?"},
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
print(text)

<|im_start|>system
你是一个专业的助手<|im_end|>
<|im_start|>user
你现在怎么样<|im_end|>
<|im_start|>assistant
<think>

</think>

I'm fine, think you. and you?<|im_end|>
<|im_start|>assistant
<think>

</think>




`LoRA`（`Low-Rank Adaptation`）训练的数据是需要经过格式化、编码之后再输入给模型进行训练的，我们需要先将输入文本编码为 `input_ids`，将输出文本编码为 `labels`，编码之后的结果是向量。我们首先定义一个预处理函数，这个函数用于对每一个样本，同时编码其输入、输出文本并返回一个编码后的字典：

In [13]:
def process_func(example):
    MAX_LENGTH = 1024 # 设置最大序列长度
    input_ids, attention_mask, labels = [], [], [] # 初始化返回值
    # 适配chat_template
    instruction = tokenizer(
        f"<s><|im_start|>system\n{example['system']}<|im_end|>\n"
        f"<|im_start|>user\n{example['instruction'] + example['input']}<|im_end|>\n"
        f"<|im_start|>assistant\n<think>\n\n</think>\n\n",
        add_special_tokens=False
    )
    response = tokenizer(f"{example['output']}", add_special_tokens=False)
    
    # 将instruction部分和response部分的input_ids拼接，
    # 并在末尾添加eos token作为标记结束的token
    input_ids = instruction["input_ids"] + response["input_ids"] + [tokenizer.pad_token_id]
    # 注意力掩码，表示模型需要关注的位置
    attention_mask = instruction["attention_mask"] + response["attention_mask"] + [1]
    # 对于instruction，使用-100表示这些位置不计算loss（即模型不需要预测这部分）
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"] + [tokenizer.pad_token_id]
    
    if len(input_ids) > MAX_LENGTH: # 超出最大序列长度截断
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    
    return {
        "inputs_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


> 函数 `process_func` 是在微调（Fine-tuning）大语言模型（LLM）时非常关键的一个**预处理步骤**。
>
> 它的核心作用是：**将人类可读的对话文本（系统提示词、用户问题、回复）转换成机器能理解的数字序列（Tensors），并设置好训练目标的“掩码（Mask）”。**
>
> ------
>
> 1. 函数的核心逻辑：数据拼接
>
> 在大模型训练中，我们通常把一整条对话拼接成一个长字符串送入模型。这个函数采用了类似 **ChatML** 的格式（常见于 Qwen 或 DeepSeek 模型）：
>
> 拼接结构如下：
>
> 1. **System**: `<|im_start|>system\n{系统指令}<|im_end|>\n`
>2. **User**: `<|im_start|>user\n{问题}<|im_end|>\n`
> 3. **Assistant Header**: `<|im_start|>assistant\n<think>\n\n</think>\n\n` (这里预留了推理空间)
> 4. **Response**: `{模型回答}`
> 5. **End**: `[PAD]` (或 EOS 结束符)
> 
> ------
>
> 2. 最终转换出的数据样子
>
> 这个函数返回了一个字典，包含三个关键的列表。为了方便理解，我们假设序列长度非常短：
>
> (1) `input_ids` (模型的输入)
>
> 这是将所有文字转换成数字后的样子。
>
> - **内容**：指令部分 + 回答部分。
>- **样子**：`[101, 532, 234, ..., 889, 2309, 102]`
> - **作用**：告诉模型完整的上下文。
> 
> (2) `attention_mask` (注意力掩码)
>
> - **内容**：和 `input_ids` 等长，由 `1` 组成。
>- **样子**：`[1, 1, 1, ..., 1]`
> - **作用**：告诉模型哪些位置是有意义的（1代表有内容，0代表是纯填充）。
> 
> (3) `labels` (训练标签 - 最关键的地方)
>
> 这是区分“学习重点”的核心。
>
> - **样子**：`[-100, -100, -100, ..., 889, 2309, 102]`
>- **逻辑**：
>   - **Instruction 部分**：全部填充为 `-100`。在深度学习（CrossEntropyLoss）中，计算损失值时会忽略 `-100`。**这意味着模型不需要练习“怎么写问题”。**
>   - **Response 部分**：保留原始的 `input_ids` 数字。**这意味着模型要练习“怎么写出正确的回答”。**
> 
> ------
>
> 3. 形象化对比图
>
> 假设我们要让模型学习：`User: 1+1?` -> `Assistant: 2`
>
> | **位置**      | **1** | **2**    | **3**  | **4**  | **5**  | **6**  |
>| ------------- | ----- | -------- | ------ | ------ | ------ | ------ |
> | **文本内容**  | `<    | im_start | >user` | `1+1?` | `<     | im_end |
> | **input_ids** | 1001  | 45       | 1002   | 1003   | **56** | 102    |
> | **labels**    | -100  | -100     | -100   | -100   | **56** | 102    |
> 
> **结论：** 模型看到 `1001, 45, 1002, 1003` 时，被要求预测出 `56`。它不会因为没预测准前面的 `1+1?` 而受罚，只会因为没写对那个 `2` 而受罚。
>
> ------
>
> 4. 几个细节注意点
>
> 1. **`<think>` 标签**：代码中手动加入了 `\n<think>\n\n</think>\n\n`。这说明你可能正在训练一个带有**推理思维链（Chain of Thought）**的模型（如 DeepSeek-R1 系列）。它引导模型在给出最终答案前先进行思考。
>2. **截断 (Truncation)**：如果对话太长（超过 1024），代码会直接切断。
> 3. **损失忽略**：`labels` 中使用 `-100` 是 PyTorch 的标准做法，用来屏蔽掉不希望计算梯度（Loss）的部分。

In [ ]:
tokenizer = ds.map(process_func, remove_colums=ds.column_names)
tokenizer